In [ ]:
def azure_translate(texts: List[str], key: str, region: str, endpoint: str) -> List[str]:
    """Translate a list of texts to Filipino using Azure Translator."""
    url = f"{endpoint.rstrip('/')}/translate"
    params = {"api-version": "3.0", "from": "en", "to": "fil"}
    headers = {
        "Ocp-Apim-Subscription-Key": key,
        "Ocp-Apim-Subscription-Region": region,
        "Content-Type": "application/json",
    }
    payload = [{"Text": text} for text in texts]

    response = requests.post(url, params=params, headers=headers, json=payload)

    try:
        response_json = response.json()
    except Exception:
        print("Non-JSON response:")
        print(response.text[:500])
        raise

    if isinstance(response_json, dict) and "error" in response_json:
        raise RuntimeError(response_json)

    try:
        translations = [item["translations"][0]["text"] for item in response_json]
        return translations
    except Exception as e:
        print("Unexpected response structure:", e)
        print(response_json)
        raise

In [ ]:
def translate_in_chunks(texts: List[str], chunk_size: int = 20) -> List[str]:
    results: List[str] = []
    for start in range(0, len(texts), chunk_size):
        chunk = texts[start : start + chunk_size]
        attempt = 0
        while True:
            try:
                translated = azure_translate(chunk, AZURE_TRANSLATOR_KEY, AZURE_TRANSLATOR_REGION, AZURE_TRANSLATOR_ENDPOINT)
                break
            except RuntimeError as err:
                err_text = str(err)
                if "429" in err_text or "request limits" in err_text:
                    backoff = min(60, 5 * (attempt + 1))
                    print(f"Hit rate limit; sleeping {backoff}s then retrying (attempt {attempt + 1})...")
                    time.sleep(backoff)
                    attempt += 1
                    if attempt >= 5:
                        raise
                else:
                    raise
        results.extend(translated)
    return results

In [ ]:
df_xlsum = pd.read_csv("../../datasets/cleaned/cleaned_xlsum.csv")
print(f"Loaded XL-Sum rows: {len(df_xlsum)}")

In [ ]:
text_list = df_xlsum["text"].to_list()
summary_list = df_xlsum["summary"].to_list()

print("Translating text ...")
text_translated = translate_in_chunks(text_list)
print("Translating summary ...")
summary_translated = translate_in_chunks(summary_list)

In [ ]:
df_xlsum["text"] = text_translated
df_xlsum["summary"] = summary_translated

output_path = "../../datasets/translated/bing/bing_translated_xlsum.csv"
df_xlsum.to_csv(output_path, index=False)
print(f"Saved translated XL-Sum to {output_path}")